# 01 - Setup (Task 1), data preparation, baseline generation (Task 2)
**Accelerator:** GPU T4 x2 · **Internet:** ON · **Inputs:** none · **Runtime:** ~30-45 min

When it finishes: **Save Version → Save & Run All (Commit)**. Later notebooks attach this notebook's output.

In [ ]:
# ==== EDIT THIS: your GitHub fork of the repository ====
REPO_URL = "https://github.com/Malik9102/small-model-alignment.git"
REPO_DIR = "/kaggle/working/small-model-alignment"

import glob, os, shutil, subprocess, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"   # live script output in the Kaggle log
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=False)

def sh(command):
    """Run a shell command, stream its output, and STOP the notebook if it fails.
    (A plain `!command` keeps going after an error, so a crashed training run would go unnoticed.)"""
    print(f"$ {command}", flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {command}")

def find_input(pattern):
    """Locate a file/dir inside attached Kaggle inputs (any dataset/notebook slug)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"Nothing matching {pattern!r} under /kaggle/input - did you attach the right notebook output?")
    if len(hits) > 1:
        print(f"[find_input] {len(hits)} matches for {pattern!r}, using {hits[0]}")
    return hits[0]

In [ ]:
# Keep Kaggle's CUDA-matched PyTorch. Do NOT reinstall torch.
sh(f"""python -m pip uninstall -y -q torchao || true""")
sh(f"""python -m pip install -q "transformers>=4.56.2,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib pytest""")
sh(f"""python -m pip install -q -e . --no-deps""")

## Task 1 - environment check (two T4s + package versions)

In [ ]:
sh(f"""python scripts/env_check.py""")

## Offline correctness tests (public smoke tests + extended edge cases + tiny end-to-end pipeline)

In [ ]:
sh(f"""PYTHONPATH=. python -m pytest -q tests/""")

## Fixed data files
SFT 30k/1k (UltraChat), HH-RLHF 15k/1k, UltraFeedback 15k/1k, AlpacaEval 300 prompts (seed 42).

In [ ]:
sh(f"""python scripts/prepare_data.py --output_dir data/processed""")
sh(f"""cat data/processed/manifest.json""")

## Task 2 - baseline generations (Qwen/Qwen2.5-0.5B base)
Sanity check on 5 prompts first, then the full sets.

In [ ]:
sh(f"""python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-base --output outputs/generations/sanity_base.json --limit 5 --print_samples --overwrite""")

In [ ]:
sh(f"""python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-base --output outputs/generations/base_alpaca.json""")
sh(f"""python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator qwen2.5-0.5b-base --output outputs/generations/base_hh.json""")

In [ ]:
sh(f"""python scripts/analyze_generations.py outputs/generations/base_alpaca.json outputs/generations/base_hh.json""")

In [ ]:
# Lightweight archive (metrics/logs/plots/generations - not the 2 GB checkpoints)
sh(f"""cd {REPO_DIR} && zip -qr /kaggle/working/01_data_and_baseline.zip data/processed eval_sets outputs/generations -x "*/best/*" "*/final/*" "*/trainer/*" && ls -lh /kaggle/working/01_data_and_baseline.zip""")
sh(f"""df -h /kaggle/working | tail -1""")